In [2]:
import pandas as pd

In [3]:
# Load the desktop and laptop data
desktop = pd.read_csv('desktop.csv')
laptop = pd.read_csv('laptop.csv')

In [4]:
# Display the first 3 rows of the desktop data
desktop.head(3)

,userid,spending,age,visits
0,1,1250,31,126
1,2,900,27,5
2,3,0,30,459


In [5]:
# Display the first 3 rows of the laptop data
laptop.head(3)

,userid,spending,age,visits
0,31,1499,32,12
1,32,799,23,40
2,33,1200,45,22


In [7]:
from scipy.stats import ttest_ind

In [8]:
# Perform a t-test on the laptop and desktop spending data
ttest_ind(desktop['spending'],laptop['spending'])

TtestResult(statistic=-2.109853741030508, pvalue=0.03919630411621095, df=58.0)

##### Observation: The desktop and laptop spending data are significantly different

In [9]:
# Perform a t-test on the desktop and latop age data
ttest_ind(desktop['age'],laptop['age'])

TtestResult(statistic=-0.7101437106800108, pvalue=0.4804606394128761, df=58.0)

##### Observation: There is not enough evidence to conclude the desktop and laptop spending data are significantly different

In [10]:
# Perform t-test on the desktop and laptop visits data
ttest_ind(desktop['visits'],laptop['visits'])

TtestResult(statistic=0.20626752311535543, pvalue=0.8373043059847984, df=58.0)

##### Observation: There is not enough evidence to conclude the desltop and laptop visits data are significantly different

## Running experiments to test new hypothesis

In [12]:
import numpy as np

In [16]:
# split the desktop sample into 2 groups by the median age of customers
# should be using train_test_split instead
medianage=np.median(desktop['age']) # calculate the median age of the desktop data
groupa=desktop.loc[desktop['age']<=medianage,:] # split data by median age
groupb=desktop.loc[desktop['age']>medianage,:]

##### Observation: Splitting the desktop data by the age value in this manner (not random) creates a confounding experiment issue

In [17]:
# Load emailresults1 data
emailresults1 = pd.read_csv('emailresults1.csv')

In [18]:
# Display the first 3 rows
emailresults1.head(3)

,userid,revenue
0,1,100
1,2,0
2,3,50


In [19]:
# merge the email results data with both the groupa and groupb split datasets
groupa_withrevenue=groupa.merge(emailresults1, on='userid')
groupb_withrevenue=groupb.merge(emailresults1, on='userid')

In [20]:
# Perform a t-test on the on the groupa and groupb data for revenue
# Groupa and groupb are split by the median age value
ttest_ind(groupa_withrevenue['revenue'], groupb_withrevenue['revenue'])

TtestResult(statistic=-2.186454851070545, pvalue=0.03730073920038287, df=28.0)

##### Observation:
 - There is significant evidence to show that the groupa and groupb revenue data are significantly different

In [21]:
# Calculate the difference between the average revenues in groupb and groupa
print(np.mean(groupb_withrevenue['revenue'])-np.mean(groupa_withrevenue['revenue']))

125.0


In [22]:
# Calculate the average revenue in groupa
print(np.mean(groupa_withrevenue['revenue']))

104.0


In [23]:
# Calculate the average revenue in groupb
print(np.mean(groupb_withrevenue['revenue']))

229.0


## Translating math into practice

In [25]:
# Randomly assign a one or zero to the laptop data in a new column called 'groupassignment1'
np.random.seed(18811015)
laptop.loc[:,'groupassignment1']=1*(np.random.random(len(laptop.index))>0.5)

# Split the laptop group into subsgroups 'groupc' and 'groupd' by the groupassignment value
groupc=laptop.loc[laptop['groupassignment1']==0,:].copy()
groupd=laptop.loc[laptop['groupassignment1']==1,:].copy()

##### Observation: By splitting the laptop data randomly, the issue of creating a confounding experiment is avoided

In [26]:
# Display the firest 3 rows
groupc.head(3)

,userid,spending,age,visits,groupassignment1
0,31,1499,32,12,0
2,33,1200,45,22,0
4,35,1350,17,85,0


In [27]:
# Display the first 3 rows
groupd.head(3)

,userid,spending,age,visits,groupassignment1
1,32,799,23,40,1
3,34,0,59,126,1
6,37,3400,65,428,1


In [28]:
# Load the emailresults2 data
emailresults2 = pd.read_csv('emailresults2.csv')

In [29]:
# Display the first 3 rows
emailresults2.head(3)

,userid,revenue
0,31,100
1,32,0
2,33,50


In [32]:
# Merge the email results data with both the groupc and groupd randomly split datasets
groupc_withrevenue=groupc.merge(emailresults2, on='userid')
groupd_withrevenue=groupd.merge(emailresults2, on='userid')

In [33]:
# Perform a t-test on the on the groupc and groupd data for revenue
# Groupc and groupd are split randomly
print(ttest_ind(groupc_withrevenue['revenue'], groupd_withrevenue['revenue']))

TtestResult(statistic=-2.381320497676198, pvalue=0.024288828555138562, df=28.0)


##### Observation: There is significant evidence to conclude that groupc and groupd revenue data are significantly different

In [35]:
# Calculate the difference between the mean values of groupd and groupc
print(np.mean(groupd_withrevenue['revenue']-np.mean(groupc_withrevenue['revenue'])))

260.3333333333333


## Understanding Effect Sizes

In [37]:
gdps=[365303000000,65994000000,220000000]

In [38]:
# Calculate the standard deviation of the 3 gdps values
np.std(gdps)

158884197328.32672

In [39]:
# Calculate the percentage 125 is to the standard deviation of gdps
125/np.std(gdps)

7.867365169217765e-10

In [42]:
burgers=[9.0,12.99,10.50]

In [43]:
# Calculate the standard deviation of the 3 burger values
np.std(burgers)

1.6455394252341695

In [44]:
# Calculate the percentage 125 is to the standard deviation of burgers
125/np.std(burgers)

75.96293232671214

## Calculating the Significance of Data

In [32]:
from statsmodels.stats.power import TTestIndPower

In [33]:
# calculate the power for the A/B test
# power is the probability that a correctly run A/B test will reject a false null hypothesis
# power should be 80% or higher to proceed
analysis = TTestIndPower()
alpha = 0.05
nobs=45
effectsize=0.5
power = analysis.solve_power(effectsize, nobs1=nobs, alpha=alpha)

In [34]:
power

0.6501855019775578

In [35]:
# calculate the number of observations needed to get to an 80% power
analysis = TTestIndPower()
alpha=0.05
power=0.8
effectsize=0.5
observations = analysis.solve_power(effect_size=effectsize, power=power, alpha=alpha)

In [36]:
print(observations)

63.7656117754095
